# Stage 6: Mô hình đa nhánh (MultiBranchFAS) - Tối ưu hóa & Huấn luyện

In [ ]:
import sys, os, gc, random, json, copy
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torchinfo import summary
import optuna

# Thêm đường dẫn tới thư mục gốc
sys.path.append(os.path.dirname(os.getcwd()))

from utils.config import Config, Params
from utils.data import get_dataloaders
from utils.train import fit
from utils.recipe import build_criterion, build_scheduler
from utils.evaluate import plot_training_history, evaluate_test_set, plot_misclassified_samples
from utils.loss import MultiBranchLoss
from models.multibranch_model import MultiBranchFAS, save_multibranch

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)

config = Config()
device = config.DEVICE
print(f"Sử dụng thiết bị: {device}")

STAGE = "multibranch"
OUT_DIR = f"../output/{STAGE}"
os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
# Lấy Dataloaders
train_loader, val_loader, test_loader, class_names = get_dataloaders(img_size=config.IMG_SIZE)
print(f"Số lượng classes: {len(class_names)} ({class_names})")

In [ ]:
# Lấy cấu hình tối ưu từ stage optimize
recipe = Params.load_recipe("optimize")
print("Recipe được load:", json.dumps(recipe, indent=2))
wd = recipe.get("weight_decay", 3.4e-6)

EPOCHS_P1 = 5
EPOCHS_P2_MINI = 5
EPOCHS_P2_FINAL = 20
N_TRIALS = 8
SEED = 42

## Pha 1: Pre-training các nhánh mới (Đóng băng Backbone)

In [ ]:
# Khởi tạo mô hình Pha 1
set_seed(SEED)
model_p1 = MultiBranchFAS(num_classes=len(class_names), pretrained=True, dropout=0.45).to(device)
model_p1.freeze_trunk()

# Setup optimizer
optimizer_p1 = optim.AdamW(model_p1.param_groups(lr_trunk=0.0, lr_branches=1e-3, lr_head=1e-3, weight_decay=wd))
criterion_p1 = MultiBranchLoss(lambda_aux=0.3)
scheduler_p1 = build_scheduler(optimizer_p1, EPOCHS_P1, {"scheduler": "cosine", "warmup_epochs": 1})

# Chạy huấn luyện Pha 1 (Mô hình trả về đã được load checkpoint tốt nhất)
model_p1, h1 = fit(model_p1, train_loader, val_loader, criterion_p1, optimizer_p1, device, class_names,
                   epochs=EPOCHS_P1, config=config, scheduler=scheduler_p1, experiment_name=f"{STAGE}/mb_p1")

# Lưu trạng thái tốt nhất của Pha 1 vào RAM để dùng cho các Trial Pha 2
best_p1_state = copy.deepcopy(model_p1.state_dict())
print("[*] Hoàn tất Pha 1 và lưu trạng thái vào RAM.")

## Pha 2 (Mini-sweep): Tìm Learning Rate phù hợp cho Mở băng toàn phần

In [ ]:
def run_phase_2(lr_base, epochs, experiment_name, seed, trial=None):
    set_seed(seed)
    model = MultiBranchFAS(num_classes=len(class_names), pretrained=True, dropout=0.45).to(device)
    
    # Nạp vạch xuất phát chuẩn từ Pha 1
    model.load_state_dict(best_p1_state)
    model.unfreeze_trunk()
    
    # Stratified LR: Trunk học siêu chậm, nhánh học vừa, head học nhanh
    lr_trunk = lr_base * 0.1
    lr_branches = lr_base * 0.6
    lr_head = lr_base
    
    optimizer = optim.AdamW(model.param_groups(lr_trunk, lr_branches, lr_head, weight_decay=wd))
    criterion = MultiBranchLoss(lambda_aux=0.1) # Pha 2 giảm nhiễu aux
    scheduler = build_scheduler(optimizer, epochs, {"scheduler": "cosine", "warmup_epochs": 1})
    
    model, history = fit(model, train_loader, val_loader, criterion, optimizer, device, class_names,
                         epochs=epochs, config=config, scheduler=scheduler,
                         experiment_name=experiment_name, trial=trial)
    return model, history

def score_of(history):
    return float(np.mean(history['val_acer'][-3:]))

def objective(trial):
    # Khảo sát Base LR
    lr_base = trial.suggest_float("lr_base", 1e-4, 2e-3, log=True)
    
    model, history = run_phase_2(lr_base, EPOCHS_P2_MINI, f"{STAGE}/trial_{trial.number}", seed=SEED + trial.number, trial=trial)
    
    # Thu gọn bộ nhớ (Tuyệt đối KHÔNG empty_cache để tránh crash Driver XPU)
    del model; gc.collect()
    
    return score_of(history)

In [ ]:
# Chạy quét tham số Pha 2
import shutil

study_name = f"{STAGE}_study"
storage_name = f"sqlite:///{OUT_DIR}/{study_name}.db"

study = optuna.create_study(study_name=study_name, storage=storage_name, load_if_exists=True, direction="minimize")

if len(study.trials) == 0:
    study.enqueue_trial({"lr_base": 5e-4}) # Mặc định cũ

study.optimize(objective, n_trials=N_TRIALS)

df = study.trials_dataframe()
display_cols = [c for c in ['number', 'value', 'state', 'params_lr_base'] if c in df.columns]
display(df[display_cols].sort_values('value').head())

# ==============================================================================
# BƯỚC RECHECK VÀ CHỌN LR AN TOÀN CHO HUẤN LUYỆN DÀI
# ==============================================================================
done_trials = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
top_trials = sorted(done_trials, key=lambda t: t.value)[:3]

recheck = []
print("\n[*] Bắt đầu recheck top 3 trials bằng seed khác...")
for rank, t in enumerate(top_trials):
    lr_base = t.params["lr_base"]
    _, h = run_phase_2(lr_base, EPOCHS_P2_MINI, f"{STAGE}/recheck_{rank}", seed=SEED + 1000 + rank)
    recheck_score = score_of(h)
    avg_score = (t.value + recheck_score) / 2
    recheck.append((t, avg_score))
    print(f"Trial {t.number} (lr={lr_base:.2e}): Lần 1 = {t.value:.4f} | Lần 2 = {recheck_score:.4f} | Trung bình = {avg_score:.4f}")
    del _ ; gc.collect()

# Tìm best score
best_score = min(r[1] for r in recheck)
good_candidates = [r for r in recheck if r[1] - best_score <= 0.01]

import math
candidate_lrs = [r[0].params["lr_base"] for r in good_candidates]
geometric_mean_lr = math.exp(sum(math.log(lr) for lr in candidate_lrs) / len(candidate_lrs))
best_lr = geometric_mean_lr / 1.5

print(f"[*] Best ACER: {best_score:.4f}")
print(f"[*] Base LR trung bình nhân: {geometric_mean_lr:.2e}")
print(f"[*] Base LR chốt cuối cùng (giảm 1.5x an toàn): {best_lr:.2e}")

## Pha 2 (Final): Huấn luyện mở băng toàn diện (20 Epochs)

In [ ]:
# Gán cứng best_lr để chạy trực tiếp nếu không chạy phần Sweep
# best_lr = 5e-4
print(f"Bắt đầu huấn luyện Pha 2 Final với Base LR: {best_lr:.2e}")

best_model_p2, history_p2 = run_phase_2(best_lr, EPOCHS_P2_FINAL, f"{STAGE}/mb_p2_final", seed=SEED)

import numpy as np
best_epoch_idx = int(np.argmin(history_p2['val_acer'])) if len(history_p2['val_acer']) > 0 else 0
best_threshold = history_p2['val_eer_threshold'][best_epoch_idx] if len(history_p2['val_eer_threshold']) > 0 else 0.5

print(f"Epoch tốt nhất: {best_epoch_idx+1} | ACER val: {history_p2['val_acer'][best_epoch_idx]:.4f} | ngưỡng: {best_threshold:.4f}")

## Đánh giá và lưu mô hình

In [ ]:
plot_training_history(history_p2, save_dir=OUT_DIR)

test_probs, test_labels, test_preds = evaluate_test_set(
    best_model_p2, val_loader, test_loader, device, class_names, 
    threshold=best_threshold, save_dir=OUT_DIR
)

plot_misclassified_samples(
    test_loader.dataset, test_probs, test_preds, test_labels, class_names, 
    num_samples=4, save_dir=OUT_DIR
)

# Lưu MỘT file .pt duy nhất (state_dict + cấu hình model + ngưỡng) để huấn luyện tiếp trên dataset khác
save_multibranch(
    best_model_p2, os.path.join(OUT_DIR, "multibranch_best.pt"), class_names, best_threshold,
    extra={"img_size": config.IMG_SIZE, "val_acer": float(history_p2['val_acer'][best_epoch_idx])}
)
print("Đã lưu:", os.path.join(OUT_DIR, "multibranch_best.pt"))

In [ ]:
OUT_MODEL_DIR = "../output/models/multibranch"
os.makedirs(OUT_MODEL_DIR, exist_ok=True)
onnx_path = os.path.join(OUT_MODEL_DIR, "multibranch_model.onnx")

best_model_p2.eval()
dummy_input = torch.randn(1, 3, config.IMG_SIZE, config.IMG_SIZE, device=device)

torch.onnx.export(
    best_model_p2, 
    dummy_input, 
    onnx_path, 
    input_names=['input'], 
    output_names=['output']
)
print(f"Đã xuất mô hình ONNX thành công tại: {onnx_path}")